## Step 0: imports
To install prerequisites execute 

`pip3 install --no-cache-dir -r initialize_requirements.txt -c constraints.txt` 

from this folder in a command prompt using the correct environment.

In [ ]:
import os
import json
from numpy.random import RandomState
from tqdm import tqdm
import re
import zipfile
import io
from datetime import datetime
import requests
from minio import Minio
import psycopg2
from PIL import Image
from io import BytesIO
from urllib.parse import urlparse
from urllib3.exceptions import MaxRetryError
import tf2onnx
import mlflow
from mlflow import pyfunc as mlflow_pyfunc
from mlflow.tracking import MlflowClient
import gdown
import tensorflow as tf 
import keras
import subprocess
import time

## Step 1: make postgress tables

### Setup minio credentials and access keys and buckets

In [ ]:
# --- 1. Configuration ---
# Local script connects to localhost
LOCAL_MINIO_ENDPOINT = "localhost:9000"
MINIO_CONTAINER_NAME = "chimp_datastore" # Must match container_name in docker-compose.yml

# Credentials from environment
MINIO_ROOT_USER = "minioadmin"
MINIO_ROOT_PASSWORD = "minioadmin"
MINIO_APP_ACCESS_KEY = "yZmhrURuUhaeVSUagMRa"
MINIO_APP_SECRET_KEY = "cnk0OxGuIgVx4La0prNaWUv7JpriCnxZfq2417ba"
MINIO_BUCKET_NAME = "manageddataset"

# Alias used by mc *inside* the container
MC_ALIAS = "minio_in_container"

def run_mc_command(*args):
    """Constructs and runs an mc command inside the MinIO container."""
    base_command = ["docker", "exec", MINIO_CONTAINER_NAME]
    mc_command = ["mc"] + list(args)
    full_command = base_command + mc_command
    
    # Setting text=True (or universal_newlines=True) is helpful for output
    return subprocess.run(full_command, capture_output=True, text=True)


def setup_mc_alias():
    """Configures the mc alias *inside* the MinIO container."""
    print(f"Setting up '{MC_ALIAS}' alias inside the container...")
    # IMPORTANT: Inside the container, MinIO is at http://localhost:9000
    result = run_mc_command(
        "alias", "set", MC_ALIAS,
        "http://localhost:9000",
        MINIO_ROOT_USER,
        MINIO_ROOT_PASSWORD,
        "--quiet"
    )
    if result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, result.args, result.stdout, result.stderr)
    print("✅ mc alias configured.")

def create_service_account():
    """Creates the service account by executing mc inside the container."""
    print(f"Checking for service account '{MINIO_APP_ACCESS_KEY}'...")
    result = run_mc_command("admin", "user", "info", MC_ALIAS, MINIO_APP_ACCESS_KEY)

    if result.returncode == 0:
        print(f"✅ Service account '{MINIO_APP_ACCESS_KEY}' already exists.")
    else:
        print(f"Service account not found. Creating '{MINIO_APP_ACCESS_KEY}'...")
        create_result = run_mc_command(
            "admin", "user", "add", MC_ALIAS,
            MINIO_APP_ACCESS_KEY,
            MINIO_APP_SECRET_KEY
        )
        if create_result.returncode != 0:
            raise subprocess.CalledProcessError(create_result.returncode, create_result.args, create_result.stdout, create_result.stderr)
        print("✅ Service account created.")

def create_bucket(client: Minio):
    """Creates the application bucket. This can still be done from the host."""
    print(f"Checking for bucket '{MINIO_BUCKET_NAME}'...")
    # ... (This function remains unchanged)
    found = client.bucket_exists(MINIO_BUCKET_NAME)
    if not found:
        print(f"Creating bucket '{MINIO_BUCKET_NAME}'...")
        client.make_bucket(MINIO_BUCKET_NAME)
        print("✅ Bucket created.")
    else:
        print(f"✅ Bucket '{MINIO_BUCKET_NAME}' already exists.")

def attach_policy_to_user():
    """Attaches the 'readwrite' policy to the application's service account."""
    policy_name = "readwrite"
    print(f"Attaching policy '{policy_name}' to user '{MINIO_APP_ACCESS_KEY}'...")
    
    # Check if the policy is already attached to prevent errors on re-runs
    # 'mc admin user info' lists the policies
    info_result = run_mc_command("admin", "user", "info", MC_ALIAS, MINIO_APP_ACCESS_KEY)
    if policy_name in info_result.stdout:
        print(f"✅ Policy '{policy_name}' is already attached.")
        return

    # Attach the policy
    attach_result = run_mc_command(
        "admin", "policy", "attach", MC_ALIAS,
        policy_name,
        f"--user={MINIO_APP_ACCESS_KEY}"
    )
    if attach_result.returncode != 0:
        raise subprocess.CalledProcessError(attach_result.returncode, attach_result.args, attach_result.stdout, attach_result.stderr)
    print("✅ Policy attached successfully.")

admin_client = Minio(
    LOCAL_MINIO_ENDPOINT,
    access_key=MINIO_ROOT_USER,
    secret_key=MINIO_ROOT_PASSWORD,
    secure=False
)

app_client = Minio(
    LOCAL_MINIO_ENDPOINT,  # Replace with your MinIO endpoint
    access_key=MINIO_APP_ACCESS_KEY,
    secret_key=MINIO_APP_SECRET_KEY,
    secure=False
)

try:
    print(" setup alias")
    setup_mc_alias()
    print(" create service account")
    create_service_account()
    print(" attach policy")
    attach_policy_to_user()
    print(" create bucket")
    create_bucket(app_client)
    print("\n🎉 MinIO setup completed successfully! 🎉")
except subprocess.CalledProcessError as e:
    print("❌ Error during MinIO setup script.")
    print(f"Command: {' '.join(e.cmd)}") # <-- FIXED
    print(f"Return Code: {e.returncode}")
    print(f"Stderr: {e.stderr}")
    print(f"Stdout: {e.stdout}")
    exit(1)
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")


In [ ]:
DATASTORE_ACCESS_KEY="yZmhrURuUhaeVSUagMRa"
DATASTORE_SECRET_KEY="cnk0OxGuIgVx4La0prNaWUv7JpriCnxZfq2417ba"
DATASTORE_URI="localhost:9000"

# Initialize MinIO client
client = Minio(
    DATASTORE_URI,  # Replace with your MinIO endpoint
    access_key=DATASTORE_ACCESS_KEY,
    secret_key=DATASTORE_SECRET_KEY,
    secure=False
)
# global bucket_name
bucket_name = "manageddataset"

if not client.bucket_exists(bucket_name):
    client.make_bucket(bucket_name)

db_config = {
    "dbname": "chimp_database",
    "user": "chimp_user",
    "password": "chimp_password",
    "host": "localhost",  
    "port": "5432"
}

conn = psycopg2.connect(**db_config)

In [ ]:
create_datapoints = """
        CREATE TABLE IF NOT EXISTS datapoints (
            id SERIAL PRIMARY KEY,
            x TEXT NOT NULL,
            y TEXT,
            metadata JSONB
        );
        """

create_labeling_tasks = """
CREATE TABLE IF NOT EXISTS labeling_tasks (
    id SERIAL PRIMARY KEY,
    dataset_id TEXT NOT NULL,
    total_images INTEGER,
    num_labeled INTEGER,
    status TEXT,
    selection JSONB
);
"""

create_dataset_query = """CREATE TABLE IF NOT EXISTS dataset (
    datapoint_id INTEGER REFERENCES datapoints(id) ON DELETE CASCADE,
    run_id TEXT NOT NULL,
    PRIMARY KEY (datapoint_id, run_id)
);"""

with conn.cursor() as cursor:
    cursor.execute(create_datapoints)
    cursor.execute(create_labeling_tasks)
    cursor.execute(create_dataset_query)
conn.commit()

## Step 2: Download base model data (model weights and training data)

In [ ]:
os.makedirs('../docker-data/files/', exist_ok=True)
modelpath = r'../docker-data/files/base_mobilenetv2_128_64_fer2013gdrive.keras'
gdown.download(f"https://drive.google.com/uc?id=1z_ZeHFk6lQLkIXyRC_6Jl1dogLYvtwkf",
               modelpath,
               quiet=False)

model = keras.models.load_model(modelpath)

mlflow.set_tracking_uri("http://localhost:8999")
name='onnx_emo_datastore'
mlflow.set_experiment(name)

with mlflow.start_run(run_name='base_mobilenetv2_128_64_fer2013'):
    # model_info = mlflow.keras.log_model(
    #     model, artifact_path="tensorflow", registered_model_name=name
    # )

    input_sig = [
            tf.TensorSpec(
                [None, 96, 96, 3],
                tf.float32,
            )
        ]
    model.output_names = ["output"]
    onnx_model, _ = tf2onnx.convert.from_keras(model, input_sig, opset=13)

    model_info = mlflow.onnx.log_model(
        onnx_model=onnx_model,
        artifact_path="model",
        registered_model_name=name
    )
    mlflow.log_artifact(modelpath, "keras")

    client = MlflowClient()
    all_versions = client.search_model_versions(f"name='{name}'")
    latest_version_obj = max(all_versions, key=lambda v: int(v.version))
    latest_version = latest_version_obj.version
    print(f"Model version {latest_version} was just logged. Transitioning to Production.")
    client.transition_model_version_stage(
        name=name,
        version=latest_version,
        stage="Production"
    )
    print(f"Successfully transitioned model version {latest_version} to 'Production'.")

    
mlflow.set_tracking_uri("http://localhost:8999")
#staging = mlflow_pyfunc.load_model(f"models:/onnx_emo_datastore/staging")
production = mlflow_pyfunc.load_model(f"models:/onnx_emo_datastore/production")

In [ ]:
zip_path = r"../docker-data/files/fer_2013gdrive.zip" #change to desired path
extract_dir = r"../docker-data/files/fer_2013gdrive"
os.makedirs(extract_dir, exist_ok=True)
gdown.download(f"https://drive.google.com/uc?id=1AN269BahDB87CiDQqUdFWwhH22oFI4oM",
               zip_path,
               quiet=False)

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

In [ ]:
directory = extract_dir + r"/fer_2013/train"
print("Looking for directory:", directory)
if not os.path.exists(directory):
    raise FileNotFoundError(f"Directory does not exist: {directory}")

username_use = "admin"
timestamp_str_use = "2024-12-19"
user_id_use = "0"

labels = []
metadata = []
zip_buffer = io.BytesIO()

with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zip_file:
    for root, _, files in os.walk(directory):
        emotion = os.path.basename(root) 
        for file in files:
            if file.endswith((".png", "jpg")): 
                file_path = os.path.join(root, file)
                # username, emotion, timestamp, user_id = parse_filename(file)
                                
                labels.append(emotion)
                metadata.append({
                    "exp": "emotion_recognition",
                    "user": username_use,
                    "userid": user_id_use,
                    "timestamp": timestamp_str_use
                })
                # Add the file to the zip
                with open(file_path, "rb") as img_file:
                    zip_file.writestr(file, img_file.read())

# Prepare the payload
timestamp_str = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
path_name = re.sub(r'[<>:"/\\|?*]', '', f"fer2013_{username_use}_{timestamp_str_use}_{user_id_use}")

files = {
    "file": (path_name + '.zip', zip_buffer.getvalue(), 'application/zip'),
    "dataset_name": (None, path_name),
    "labels": ('labels.json', json.dumps(labels), 'application/json'),
    "metadata": ('metadata.json', json.dumps(metadata), 'application/json')
}

# Send the request
url = "http://localhost:5253/managed_datasets"  
print("Sending image zip to managed dataset:", path_name)
response = requests.post(url, files=files)

print("Response status:", response.status_code)
print("Response body:", response.text)

## Make the dataset entry for the base model

In [ ]:
DATASTORE_ACCESS_KEY="yZmhrURuUhaeVSUagMRa"
DATASTORE_SECRET_KEY="cnk0OxGuIgVx4La0prNaWUv7JpriCnxZfq2417ba"
DATASTORE_URI="localhost:9000"

# Initialize MinIO client
client = Minio(
    DATASTORE_URI,  # Replace with your MinIO endpoint
    access_key=DATASTORE_ACCESS_KEY,
    secret_key=DATASTORE_SECRET_KEY,
    secure=False
)
# global bucket_name
bucket_name = "manageddataset"

# if not client.bucket_exists(bucket_name):
#     client.make_bucket(bucket_name)

db_config = {
    "dbname": "chimp_database",
    "user": "chimp_user",
    "password": "chimp_password",
    "host": "localhost",  # Use the Docker host's IP if not running locally
    "port": "5432"
}

In [ ]:
client = MlflowClient()
name = "onnx_emo_datastore"

# Get all model versions for the registered model
all_versions = client.search_model_versions(f"name='{name}'")

# Find the version in Production stage
production_version = next(
    (v for v in all_versions if v.current_stage == "Production"),
    None
)

if production_version:
    run_id = production_version.run_id
    print(f"The run_id of the Production model is: {run_id}")
else:
    print("No model version is in Production stage.")


In [ ]:
conn = psycopg2.connect(**db_config)
cursor = conn.cursor()

production = mlflow_pyfunc.load_model(f"models:/onnx_emo_datastore/production")
production.metadata
# Insert query
select_query = """
SELECT * FROM datapoints
WHERE x LIKE '%fer2013%';
"""

cursor.execute(select_query)

# Fetch all rows
rows = cursor.fetchall()

insert_data = []

for row in rows:
    datapoint_id = row[0]
    insert_data.append((datapoint_id, run_id)) 

insert_query = """INSERT INTO dataset (datapoint_id, run_id) VALUES (%s, %s);"""

In [ ]:
cursor.executemany(insert_query, insert_data)
conn.commit()